# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boda-25/FlyRank-machine-learning/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


### Selected Lane: Lane 2 — Refresh / Content Opportunity Scoring

**Rule explanation:**

I will prioritize pages that have not been updated for at least 180 days and have received at least 500 impressions during the last 90 days.

The score will use impressions as a measure of visibility, but only pages meeting both conditions will receive a positive score. Pages that do not meet the conditions will receive a score of zero.

**Reason codes:**
- `stale_visible_page`: The page is at least 180 days since its last update and has at least 500 impressions.
- `below_rule_threshold`: The page does not meet both conditions.

**Action labels:**
- `REVIEW_FOR_REFRESH`: Send the page for human review.
- `MONITOR_ONLY`: Do not prioritize it for refreshing under this rule.

**Important:** This is a simple rule-based baseline, not a prediction of future traffic or guaranteed improvement.


In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

# Load the real starter dataset
DATA_URL = (
    "https://raw.githubusercontent.com/Boda-25/"
    "FlyRank-machine-learning/main/data/raw/"
    "content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)
print("Unique content items:", df["content_id"].nunique())

# Convert the two selected signals to numeric
df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"], errors="coerce"
)
df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"], errors="coerce"
)

# Signal 1: Content staleness
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, 365, np.inf],
    labels=["0-30", "31-90", "91-180", "181-365", "365+"]
).astype("string").fillna("missing")

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_impressions=("impressions_90d", "median"),
        visible_pages_n=("impressions_90d",
                         lambda x: (x >= 500).sum())
    )
)

print("\nSIGNAL 1 — CONTENT STALENESS")
display(staleness_table)

stale_candidates = (
    (df["days_since_last_update"] >= 180).sum()
)

stale_verdict = (
    "CONFIRMED" if stale_candidates >= 20
    else "MIXED" if stale_candidates > 0
    else "FALSE"
)

print("Staleness verdict:", stale_verdict)
print("Pages at least 180 days since update:", stale_candidates)

# Signal 2: Search visibility / impression volume
df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-np.inf, 500, 3000, 10000, np.inf],
    labels=["Below 500", "500-2999", "3000-9999", "10000+"]
).astype("string").fillna("missing")

volume_table = (
    df.groupby("volume_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_days_since_update=("days_since_last_update", "median"),
        stale_pages_n=("days_since_last_update",
                       lambda x: (x >= 180).sum())
    )
)

print("\nSIGNAL 2 — SEARCH VISIBILITY")
display(volume_table)

visible_candidates = (
    (df["impressions_90d"] >= 500).sum()
)

volume_verdict = (
    "CONFIRMED" if visible_candidates >= 20
    else "MIXED" if visible_candidates > 0
    else "FALSE"
)

print("Visibility verdict:", volume_verdict)
print("Pages with at least 500 impressions:", visible_candidates)

# Check the intersection used by the rule
eligible_count = (
    (df["days_since_last_update"] >= 180) &
    (df["impressions_90d"] >= 500)
).sum()

print("\nPages meeting BOTH rule conditions:", eligible_count)


Dataset shape: (30000, 44)
Unique content items: 30000

SIGNAL 1 — CONTENT STALENESS


,n,median_impressions,visible_pages_n
staleness_bucket,,,
0-30,20480,470.0,10063
181-365,169,16.0,17
31-90,175,510.0,88
365+,5,2.0,0
91-180,9171,1692.0,6558


Staleness verdict: CONFIRMED
Pages at least 180 days since update: 174

SIGNAL 2 — SEARCH VISIBILITY


,n,median_days_since_update,stale_pages_n
volume_bucket,,,
10000+,3602,25.0,4
3000-9999,4681,22.0,5
500-2999,8432,22.0,8
Below 500,13285,20.0,157


Visibility verdict: CONFIRMED
Pages with at least 500 impressions: 16726

Pages meeting BOTH rule conditions: 17


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:

# Define the baseline rule using observed signals only

stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500

# Both conditions must be true
df["rule_eligible"] = stale & visible

# Transparent score: impressions for eligible pages, zero otherwise
df["baseline_score"] = np.where(
    df["rule_eligible"],
    df["impressions_90d"],
    0
)

# One reason code per page
df["reason_code"] = np.where(
    df["rule_eligible"],
    "stale_visible_page",
    "below_rule_threshold"
)

# Action label
df["action_label"] = np.where(
    df["rule_eligible"],
    "REVIEW_FOR_REFRESH",
    "MONITOR_ONLY"
)

# Create the ranked queue
queue = df[
    [
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].copy()

queue = queue.sort_values(
    by=["baseline_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

queue.insert(0, "rank", range(1, len(queue) + 1))

# Save the output
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Baseline queue created successfully!")
print("Output:", output_path)
print("Total ranked pages:", len(queue))
print("Pages recommended for review:", int(df["rule_eligible"].sum()))
print("Pages not prioritized:", int((~df["rule_eligible"]).sum()))

display(queue.head(10))


Baseline queue created successfully!
Output: work/outputs/baseline_action_score.csv
Total ranked pages: 30000
Pages recommended for review: 17
Pages not prioritized: 29983


,rank,content_id,days_since_last_update,impressions_90d,baseline_score,reason_code,action_label
0,1,content_cf56e2e2e282,194,61678,61678,stale_visible_page,REVIEW_FOR_REFRESH
1,2,content_7368877ea310,194,59472,59472,stale_visible_page,REVIEW_FOR_REFRESH
2,3,content_1bfaa38ff26c,194,25715,25715,stale_visible_page,REVIEW_FOR_REFRESH
3,4,content_0a91db491d14,193,13299,13299,stale_visible_page,REVIEW_FOR_REFRESH
4,5,content_5feee3994adb,194,7812,7812,stale_visible_page,REVIEW_FOR_REFRESH
5,6,content_c2d929d83eaa,193,7558,7558,stale_visible_page,REVIEW_FOR_REFRESH
6,7,content_b16bd7307b39,194,4590,4590,stale_visible_page,REVIEW_FOR_REFRESH
7,8,content_fe16a55cd13d,194,4556,4556,stale_visible_page,REVIEW_FOR_REFRESH
8,9,content_ecb6215e79fd,194,4429,4429,stale_visible_page,REVIEW_FOR_REFRESH
9,10,content_928af3e22c80,193,1697,1697,stale_visible_page,REVIEW_FOR_REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:

# Select the top 20 pages
top20 = queue.head(20).copy()

def confidence_note(row):
    impressions = row["impressions_90d"]
    age = row["days_since_last_update"]

    if impressions >= 3000 and age >= 365:
        return "High relative confidence — strong visibility and old update"
    elif impressions >= 1000 and age >= 210:
        return "Medium relative confidence — sufficient visibility"
    else:
        return "Low relative confidence — threshold or volume is borderline"

def wrong_reason(row):
    impressions = row["impressions_90d"]
    age = row["days_since_last_update"]

    if impressions < 1000:
        return (
            "The impression volume is close to the minimum threshold; "
            "the opportunity may be noisy."
        )
    elif age < 210:
        return (
            "The page is close to the staleness threshold; "
            "it may not actually need a refresh."
        )
    else:
        return (
            "Seasonality, changing search demand, or a related page "
            "absorbing traffic could explain the situation."
        )

top20["confidence_note"] = top20.apply(
    confidence_note, axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    wrong_reason, axis=1
)

print("TOP-20 MANUAL REVIEW")
print("Each row requires human verification.")

review_columns = [
    "rank",
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    "baseline_score",
    "reason_code",
    "action_label",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20[review_columns].to_string(index=False))


TOP-20 MANUAL REVIEW
Each row requires human verification.


' rank           content_id  days_since_last_update  impressions_90d  baseline_score          reason_code       action_label                                             confidence_note                                                               what_would_make_it_wrong\n    1 content_cf56e2e2e282                     194            61678           61678   stale_visible_page REVIEW_FOR_REFRESH Low relative confidence — threshold or volume is borderline      The page is close to the staleness threshold; it may not actually need a refresh.\n    2 content_7368877ea310                     194            59472           59472   stale_visible_page REVIEW_FOR_REFRESH Low relative confidence — threshold or volume is borderline      The page is close to the staleness threshold; it may not actually need a refresh.\n    3 content_1bfaa38ff26c                     194            25715           25715   stale_visible_page REVIEW_FOR_REFRESH Low relative confidence — threshold or volume is borderline

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


### Weak Picks and Leakage Check

Some recommendations may be weak even when they satisfy the rule.

- Pages with impressions close to 500 may have noisy or unstable demand.
- Pages close to the 180-day update threshold may not need an immediate refresh.
- Seasonal changes or related pages may explain some performance patterns.

I will treat the ranked queue as a review recommendation, not an automatic instruction to edit content.

**Leakage prevention:**

The baseline uses only two observed signals:
- `days_since_last_update`
- `impressions_90d`

It does not use product-generated flags, `trend_direction`, `trend_pct`, target labels, or future-window metrics.

The score is calculated only from the two selected signals.


In [4]:

# Identify five potentially weak recommendations from the top 20

weak_picks = top20.nsmallest(5, "baseline_score").copy()

def explain_weak_pick(row):
    if row["action_label"] == "MONITOR_ONLY":
        return "Does not meet both rule thresholds; should not be treated as a refresh recommendation."

    if row["impressions_90d"] < 1000:
        return "Low visibility above the minimum threshold; the opportunity may be noisy."

    if row["days_since_last_update"] < 210:
        return "Close to the 180-day cutoff; an update may not be necessary yet."

    return "Seasonality, search-demand changes, or related-page overlap should be investigated."

weak_picks["weakness_reason"] = weak_picks.apply(
    explain_weak_pick, axis=1
)

print("POTENTIALLY WEAK PICKS")
display(
    weak_picks[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action_label",
            "weakness_reason"
        ]
    ]
)

# Leakage audit
rule_features = [
    "days_since_last_update",
    "impressions_90d"
]

forbidden_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
    "needs_ctr_fix",
    "is_quick_win"
]

leaked_features = set(rule_features).intersection(
    forbidden_columns
)

assert len(leaked_features) == 0, (
    f"Leakage detected: {leaked_features}"
)

print("\nLEAKAGE AUDIT")
print("Rule features:", rule_features)
print("Forbidden features used:", list(leaked_features))
print("Product flags used: None")
print("Future-window features used: None")
print("Target/label-derived features used: None")
print("Leakage audit passed!")


POTENTIALLY WEAK PICKS


,rank,content_id,baseline_score,reason_code,action_label,weakness_reason
17,18,content_5fe46e04994d,0,below_rule_threshold,MONITOR_ONLY,Does not meet both rule thresholds; should not...
18,19,content_aaef01a50def,0,below_rule_threshold,MONITOR_ONLY,Does not meet both rule thresholds; should not...
19,20,content_8c19996aa890,0,below_rule_threshold,MONITOR_ONLY,Does not meet both rule thresholds; should not...
16,17,content_074ba6ead17b,533,stale_visible_page,REVIEW_FOR_REFRESH,Low visibility above the minimum threshold; th...
15,16,content_6226ee6adc91,545,stale_visible_page,REVIEW_FOR_REFRESH,Low visibility above the minimum threshold; th...



LEAKAGE AUDIT
Rule features: ['days_since_last_update', 'impressions_90d']
Forbidden features used: []
Product flags used: None
Future-window features used: None
Target/label-derived features used: None
Leakage audit passed!



## Self-check

- [ ] Both signals were checked using bucket tables with visible sample counts (n).
- [ ] Signal verdicts were printed and explained.
- [ ] One transparent baseline rule was implemented.
- [ ] Every page has a score, reason code, and action label.
- [ ] The ranked queue was generated and saved to `work/outputs/baseline_action_score.csv`.
- [ ] All top 20 pages were reviewed with action, reason, confidence, and what could make the recommendation wrong.
- [ ] Weak picks were identified and explained.
- [ ] Leakage checks passed: no product flags, future-window inputs, or target-derived features were used.
- [ ] The notebook runs from top to bottom without errors.
- [ ] No client names, raw URLs, or private queries are exposed.
- [ ] The notebook was committed to `work/notebooks/w04_baseline_score.ipynb`.
- [ ] The repository URL was submitted on the assignment card.
